# 02.P · Mini-project: predict customer spend (scaffold)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/02-regression/project/project.ipynb)

Brief and rubric: [README.md](README.md). Features from before 1 Sep 2011; target = spend 1 Sep – 30 Nov 2011.

## 1. Build the customer table

Clean the transactions and aggregate one row per customer. Report the number of customers and the share with zero target.

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

raw = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
print(raw.shape)
raw.head()


def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df.copy()
    out = out[~out["Invoice"].astype(str).str.startswith("C")]   # drop cancellations
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)]         # drop returns / free items
    out = out.dropna(subset=["Customer ID"])                       # drop guest checkouts
    out = out.drop_duplicates()
    out["Customer ID"] = out["Customer ID"].astype(int)
    out["Revenue"] = out["Quantity"] * out["Price"]
    return out

sales = clean_retail(raw)
print(f"{len(raw):,} rows → {len(sales):,} after cleaning")


cutoff = pd.Timestamp("2011-09-01")
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(
    orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
    products=("StockCode", "nunique"), items=("Quantity", "sum"),
    first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"),
    country=("Country", "first"))
cust["avg_order"] = cust["spend"] / cust["orders"]
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
cust["spend_next_quarter"] = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust = cust.drop(columns=["first_order", "last_order"])
print(f"{len(cust):,} customers; features from before {cutoff.date()}, target = spend Sep–Nov 2011")
cust.head()

(541910, 8)
541,910 rows → 392,693 after cleaning


3,317 customers; features from before 2011-09-01, target = spend Sep–Nov 2011


,orders,spend,products,items,country,avg_order,tenure_days,days_since_last,spend_next_quarter
Customer ID,,,,,,,,,
12346,1,77183.60,1,74215,United Kingdom,77183.600000,225,225,0.00
12347,5,2790.86,82,1590,Iceland,558.172000,267,29,1294.32
12348,3,1487.24,22,2124,Finland,495.746667,258,148,310.00
12350,1,334.40,17,197,Norway,334.400000,210,210,0.00
12352,5,1561.81,26,254,Norway,312.362000,196,162,944.23


In [2]:
# TODO: your code here
# share of customers with spend_next_quarter == 0

## 2. Hold out a test set (20%, random_state=42) before modelling

In [3]:
# TODO: your code here
# feats = [...]; X_tr, X_te, y_tr, y_te = train_test_split(...)

## 3. Baselines: mean and median

In [4]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def report(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name:<28} MAE £{mae:8.1f}   RMSE £{rmse:8.1f}   R² {r2:6.3f}")
# TODO: your code here

## 4. One feature: a line on `spend`

In [5]:
# TODO: your code here

## 5. All six features: multiple linear regression

Print the coefficients. Explain two of them in a markdown cell, including any surprising sign.

In [6]:
# TODO: your code here

*Your interpretation here.*

## 6. A curve: polynomial degree 2 on `spend`

In [7]:
# TODO: your code here
# make_pipeline(StandardScaler(), PolynomialFeatures(2), LinearRegression())

## 7. Trees: max_depth in {2, 4, 8, None}, train R² vs test R²

In [8]:
# TODO: your code here

*Where does overfitting start?*

## 8. Leaderboard

In [9]:
# TODO: your code here
# one DataFrame: model, test MAE, test RMSE, test R²

*Which metric should marketing care about, and why?*

## 9. Residuals of the winning model; the three biggest misses

In [10]:
# TODO: your code here

## 10. The ranked list: top 20 customers by predicted next-quarter spend

In [11]:
# TODO: your code here

## 11. Recommendation to the head of marketing

*Model, typical error in pounds, caveat about wholesalers, next step.*